# Global Pollution Analysis and Energy Recovery

Classification of countries into **Low, Medium, High Pollution Severity** using:
- Multinomial Naive Bayes
- K-Nearest Neighbors (KNN)
- Decision Tree

Dataset: Global_Pollution_Analysis 1.csv


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, precision_score,
    recall_score, f1_score
)


In [ ]:

df = pd.read_csv('Global_Pollution_Analysis 1.csv')
df.head()


In [ ]:

print(df.shape)
df.info()
df.isnull().sum()


## Data Cleaning & Missing Value Handling

In [ ]:

num_cols = df.select_dtypes(include=np.number).columns

imputer = SimpleImputer(strategy='median')
df[num_cols] = imputer.fit_transform(df[num_cols])

df.head()


## Feature Engineering

In [ ]:

df['Pollution_Score'] = (
    df['Air_Pollution_Index'] +
    df['Water_Pollution_Index'] +
    df['Soil_Pollution_Index']
) / 3

df['Energy_Per_Person'] = (
    df['Energy_Recovered (in GWh)'] /
    df['Population (in millions)']
)

df['Pollution_Trend'] = (
    df.groupby('Country')['Pollution_Score']
      .transform(lambda x: x - x.mean())
)

df.head()


In [ ]:

df['Pollution_Severity'] = pd.qcut(
    df['Pollution_Score'],
    q=3,
    labels=['Low','Medium','High']
)

df['Pollution_Severity'].value_counts()


## Encoding

In [ ]:

country_encoder = LabelEncoder()
df['Country'] = country_encoder.fit_transform(df['Country'])

severity_encoder = LabelEncoder()
df['Pollution_Severity'] = severity_encoder.fit_transform(
    df['Pollution_Severity']
)


## Visualization

In [ ]:

plt.figure(figsize=(8,5))
sns.histplot(df['Pollution_Score'], kde=True)
plt.title('Pollution Score Distribution')
plt.show()


In [ ]:

plt.figure(figsize=(8,6))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


## Feature Scaling

In [ ]:

X = df.drop(columns=['Pollution_Severity'])
y = df['Pollution_Severity']

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:

def evaluate_model(model, X_test, y_test, name):

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(name)
    print('Accuracy:', acc)
    print('Precision:', prec)
    print('Recall:', rec)
    print('F1:', f1)

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f'Confusion Matrix - {name}')
    plt.show()

    print(classification_report(y_test, y_pred))

    return [name, acc, prec, rec, f1]


## Multinomial Naive Bayes

In [ ]:

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

nb_results = evaluate_model(
    nb_model,
    X_test,
    y_test,
    'Multinomial Naive Bayes'
)


## KNN Classifier

In [ ]:

param_grid = {'n_neighbors':[3,5,7,9,11]}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy'
)

knn_grid.fit(X_train, y_train)

print('Best Parameters:', knn_grid.best_params_)

knn_model = knn_grid.best_estimator_

knn_results = evaluate_model(
    knn_model,
    X_test,
    y_test,
    'KNN'
)


## Decision Tree

In [ ]:

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {
        'max_depth':[3,5,7,10,None],
        'min_samples_split':[2,5,10]
    },
    cv=5,
    scoring='accuracy'
)

dt_grid.fit(X_train, y_train)

print('Best Parameters:', dt_grid.best_params_)

dt_model = dt_grid.best_estimator_

dt_results = evaluate_model(
    dt_model,
    X_test,
    y_test,
    'Decision Tree'
)


## Model Comparison

In [ ]:

results = pd.DataFrame(
    [nb_results, knn_results, dt_results],
    columns=[
        'Model',
        'Accuracy',
        'Precision',
        'Recall',
        'F1 Score'
    ]
)

results.sort_values(
    by='Accuracy',
    ascending=False
)



## Actionable Insights

1. Countries with higher air, water, and soil pollution indices are classified into High Pollution Severity.
2. Energy recovery and renewable energy adoption can help offset environmental impact.
3. KNN often performs well when pollution patterns are clustered.
4. Decision Trees provide better interpretability for policy-making.
5. Governments should invest in renewable energy and industrial waste management to reduce pollution.
